In [ ]:
import geopandas as gpd
import pandas as pd
from pathlib import Path

In [ ]:
input_dir = Path.cwd() / "input_kansen_db_liwo"
output_dir = Path.cwd() / "output_kansen_db_liwo"

In [ ]:
primair = input_dir / "Primaire_keringen" / "geo_infrastructuur_primaire_keringen.shp"
niet_primair = (
    input_dir
    / "Niet_primaire_keringen"
    / "geo_infrastructuur_niet_primaire_keringen.shp"
)
doorbraak_locaties_primair = (
    input_dir
    / "Doorbraaklocaties"
    / "geo_gebiedsindeling_doorbraaklocaties_primair.shp"
)
doorbraak_locaties_regionaal = (
    input_dir
    / "Doorbraaklocaties"
    / "geo_gebiedsindeling_doorbraaklocaties_regionaal.shp"
)

geometrie_35_3 = (
    input_dir
    / "Geometrie_voor_Traject_#35-3"
    / "Normtrajecten_trajectdelen_v26092022.shp"
)

In [ ]:
scenario_list = pd.read_excel(input_dir / "scenariolist.xlsx")
scenario_manager = pd.read_excel(
    input_dir / "scenmanager_compare.xlsx", sheet_name="kansen_trajecten_aangepast"
)

In [ ]:
gdf_primair = gpd.read_file(primair)
gdf_niet_primair = gpd.read_file(niet_primair)
gdf_doorbraak_locaties_primair = gpd.read_file(doorbraak_locaties_primair)
gdf_doorbraak_locaties_regionaal = gpd.read_file(doorbraak_locaties_regionaal)
gdf_geometrie_35_3 = gpd.read_file(geometrie_35_3)

gdf_primair.set_index(["TRAJECT_ID", "TRAJECT_DL"], inplace=True)

Klein zijspoor voor traject 35-3, hier is de geometries verloren gegaan. In de oude versie ging dit om 36-0 maar dat is dezelfde als 36-1

In [ ]:
set(scenario_manager.set_index(["TRAJECT_ID", "TRAJECT_DL"]).index).difference(
    set(gdf_primair.index)
)

In [ ]:
print(
    scenario_manager.set_index(["TRAJECT_ID", "TRAJECT_DL"]).loc[
        ("#35-3", "036013"), "WIJZ_UPDATE_2022"
    ]
)

In [ ]:
scenario_manager.set_index(["TRAJECT_ID", "TRAJECT_DL"]).loc[("#35-3", "036013")][
    ["LENGTE_DL", "KANSDL_2022"]
]

In [ ]:
gdf_36_0 = gdf_geometrie_35_3[gdf_geometrie_35_3["TRAJECT_ID"] == "36-0"].set_index(
    ["TRAJECT_ID", "TRAJECT_DL"]
)

gdf_36_0_adj = gdf_36_0[
    ["LENGTE_TRA", "DUIDING205", "KLASSE2050", "KANS2019", "geometry"]
].copy()
gdf_36_0_adj.rename(
    columns={
        "LENGTE_TRA": "Shape_Leng",
        "DUIDING205": "DUIDING",
        "KLASSE2050": "klasse",
        "KANS2019": "KANS",
    },
    inplace=True,
)

# neem de lans over uit de scenario manager
gdf_36_0_adj.loc[("36-0", "036013"), "KANS"] = scenario_manager.set_index(
    ["TRAJECT_ID", "TRAJECT_DL"]
).loc[("#35-3", "036013")]["KANSDL_2022"]

gdf_36_0_adj.index = pd.MultiIndex.from_tuples(
    [("#35-3", "036013")], names=["TRAJECT_ID", "TRAJECT_DL"]
)
gdf_36_0_adj

In [ ]:
gdf_primair_comb = gpd.GeoDataFrame(pd.concat([gdf_primair, gdf_36_0_adj]))

In [ ]:
df_kansen_2025 = scenario_manager.set_index(["TRAJECT_ID", "TRAJECT_DL"])["NORMOG_2050"]
gdf_primair_kansen = gdf_primair_comb.join(df_kansen_2025, how="inner")

In [ ]:
gdf_primair_kansen

In [ ]:
# voor een aantal locaties missen we nog wat dingen
gdf_primair_kansen["KANS_2100"] = 1

In [ ]:
layers = [
    gdf_primair_kansen,
    gdf_niet_primair,
    gdf_doorbraak_locaties_primair,
    gdf_doorbraak_locaties_regionaal,
]

drop_list = [
    "gid",
    "id",
    "geom",
    "notify",
    "LT30",
    "F30T300",
    "F300T3000",
    "F3000T30K",
    "GT30K",
    "sign",
    "KENMERK_V2",
    "keuze",
    "cascade",
]
rename_dict = {
    "NORMOG_2050": "KANS_2050",
    "Opmerkinge": "Opmerkingen",
    "kansk": "klasse_actueel",
    "klasse": "klasse_actueel",
    "Shape_Leng": "length",
}
for layer in layers:
    for col in drop_list:
        if col in layer.columns:
            layer.drop(columns=[col], inplace=True)
    layer.rename(columns=rename_dict, inplace=True)

In [ ]:
display(gdf_primair_kansen.head())
display(gdf_niet_primair.head())
display(gdf_doorbraak_locaties_primair.head())
display(gdf_doorbraak_locaties_regionaal.head())
# display(gdf_primair.head())

In [ ]:
driver = "GPKG"
gdf_primair_kansen.to_file(
    output_dir / "kansendatabase.gpkg", driver=driver, layer="primaire_keringen"
)
gdf_niet_primair.to_file(
    output_dir / "kansendatabase.gpkg", driver=driver, layer="niet_primaire_keringen"
)
gdf_doorbraak_locaties_primair.to_file(
    output_dir / "kansendatabase.gpkg",
    driver=driver,
    layer="doorbraak_locaties_primair",
)
gdf_doorbraak_locaties_regionaal.to_file(
    output_dir / "kansendatabase.gpkg",
    driver=driver,
    layer="doorbraak_locaties_regionaal",
)

In [ ]:
driver = "GeoJSON"
gdf_primair_kansen.to_file(
    output_dir / "kansendatabase_primaire_keringen.geojson",
    driver=driver,
)
gdf_niet_primair.to_file(
    output_dir / "kansendatabase_niet_primaire_keringen.geojson",
    driver=driver,
)
gdf_doorbraak_locaties_primair.to_file(
    output_dir / "kansendatabase_doorbraak_locaties_primair.geojson",
    driver=driver,
)
gdf_doorbraak_locaties_regionaal.to_file(
    output_dir / "kansendatabase_doorbraak_locaties_regionaal.geojson",
    driver=driver,
)